# 🎥 Real Text-to-Video Test — Wan2.1 T2V 1.3B

This notebook now uses **Wan2.1 T2V 1.3B**, a real text-to-video model. It generates motion directly from text — not a still image with camera animation.

Test goal: a real human walks naturally, feet/legs and arms move, clothing moves, the camera physically follows, and the scene contains a reveal.

**GPU:** Kaggle T4/T4x2. The official Wan2.1 project states the T2V-1.3B model is designed for consumer GPUs and supports 480P text-to-video. We use CPU offloading and keep the text encoder on CPU to reduce VRAM use.

In [ ]:
!nvidia-smi
!rm -rf /kaggle/working/LTX-Video /kaggle/working/models
!git clone -q --depth 1 https://github.com/Wan-Video/Wan2.1.git /kaggle/working/Wan2.1
!pip -q install opencv-python diffusers transformers tokenizers accelerate tqdm imageio easydict ftfy imageio-ffmpeg dashscope 'numpy<2'\n!pip -q install flash-attn --no-build-isolation\n!sudo apt-get update -qq && sudo apt-get install -y -qq ffmpeg
%cd /kaggle/working/Wan2.1

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_DIR = Path('/kaggle/working/Wan2.1-T2V-1.3B')
MODEL_DIR.mkdir(exist_ok=True)

snapshot_download(
    repo_id='Wan-AI/Wan2.1-T2V-1.3B',
    local_dir=MODEL_DIR,
    allow_patterns=[
        '*.safetensors', '*.pth', '*.json', '*.model', '*.txt'
    ],
)
print('Model ready:', MODEL_DIR)


## 🎬 Actual action + camera prompt

The prompt is deliberately chronological so the model has explicit human motion and physical camera movement.

In [ ]:
PROMPT = '''Photorealistic live-action cinematic video, vertical composition. A real young man wearing a dark jacket and jeans walks naturally down an abandoned underground railway platform. His feet visibly take continuous alternating steps, his knees bend naturally, his arms swing with every step, his shoulders and body weight shift realistically, and his jacket moves with his walking. The same man remains consistent throughout the shot. The camera starts several meters behind and slightly above him, then physically descends and smoothly tracks forward at his walking speed. The camera passes close to concrete pillars and foreground objects, creating strong real parallax and changing perspective. The man notices a faint warm light ahead, slows down, and approaches a dark maintenance doorway. The camera follows directly behind him, then smoothly moves around his shoulder as he reaches the doorway. He opens the heavy door and reveals a gigantic hidden underground city far below, filled with distant lights, roads and enormous structures. He naturally stops in surprise while the camera continues moving forward past him toward the massive reveal. Continuous real human motion, realistic feet and hands, believable physics, natural cloth motion, realistic shadows, cinematic depth, natural motion blur, premium live-action photography, realistic lens behavior, no slideshow, no static image, no digital zoom, no frozen body, no CGI-looking human, no cartoon, no text, no logo, no watermark.'''
print(PROMPT)

In [ ]:
import subprocess, shlex, os

raw='/kaggle/working/wan_real_t2v_test_raw.mp4'
final='/kaggle/working/wan_real_t2v_test_9x16.mp4'

cmd=[
  'python','generate.py',
  '--task','t2v-1.3B',
  '--size','480*832',
  '--frame_num','81',
  '--sample_steps','25',
  '--sample_shift','8',
  '--sample_guide_scale','6',
  '--ckpt_dir','/kaggle/working/Wan2.1-T2V-1.3B',
  '--offload_model','True',
  '--t5_cpu',
  '--base_seed','12345',
  '--prompt',PROMPT,
  '--save_file',raw,
]
print('Running Wan2.1 T2V 1.3B...')
print('Command:', ' '.join(shlex.quote(x) for x in cmd if x != PROMPT))
subprocess.run(cmd, check=True)

# Crop 480x832 to an exact 9:16 vertical frame: 468x832.
ff=[
  'ffmpeg','-y','-i',raw,
  '-vf','crop=468:832:(in_w-468)/2:0,setsar=1',
  '-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',final,
]
subprocess.run(ff, check=True)
print('DONE:', final, os.path.getsize(final))

In [ ]:
from IPython.display import Video, display
display(Video('/kaggle/working/wan_real_t2v_test_9x16.mp4', embed=True, width=360))